**MONITORAMENTO E PREDIÇÃO CLIMÁTICA (ARQUITETURA ZARR & ETL)**
Pipeline ETL (1990-2025) para gerar dataset Cloud-Optimized (ARCO) focado em ML preditivo de chuvas no NE.

1. DOMÍNIOS (WGS84): Preditores/Atlântico (30°N-30°S | 100°W-20°E); Alvo Macro/ENEB (2°S-12°S | 34.5°W-40°W); Alvo Micro/Recife (7.5°S-8.5°S | 34.7°W-35.5°W).

2. PIPELINE: Extração API Copernicus (ERA5/ORAS5) c/ auditoria de corrupção -> Transformação via Xarray (somas/médias diárias) -> Load particionado em Zarr com Self-Cleaning (exclusão imediata do cache NetCDF p/ poupar disco).

3. VARIÁVEIS (20): ERA5 Sup. (Precip, T2m, TSM/SST, Pressão MSL, Ventos-10m [U e V], CAPE, Fluxos [Latente/Sensível], Água Prec.); ERA5 Alt. (Geopot, Umidade, Ômega @ 850/500/200hPa); ORAS5 Oceano (TSM/SST, Salinidade, SSH, MLD, OHC).

4. ENG. DADOS E IA: 
- Features: Deltas temporais (tendências), Encoding Cíclico (seno/cosseno p/ tempo/ventos), e Interações (ex: Índice de Calor).
- Estruturação: Janelas Deslizantes (lookback 3D p/ redes LSTM) e K-Means (clusters de regimes climáticos).
- Predição/Explainability: XGBoost/RF com Class Weights (foco em chuvas/minoria) + SHAP values para mapear lags oceânicos e feature importance.

In [ ]:
import os
import cdsapi
import xarray as xr
import pandas as pd
import time
import zipfile

# --- MENSAGEM DE INICIALIZAÇÃO ---
print("="*80, flush=True)
print("Iniciando o pipeline de dados atmosféricos e oceânicos...", flush=True)
print("conectando ao sistema copernicus.....", flush=True)
time.sleep(1) # Pequena pausa de 1 segundo para visualização no terminal

# MÓDULO 1: CONFIGURAÇÕES E CLIENTE
# ------------------------------------------------------------------------------
CDS_URL = "https://cds.climate.copernicus.eu/api"
CDS_KEY = "e070e6f5-c003-4a24-8d53-c40d433402cd"

try:
    c = cdsapi.Client(url=CDS_URL, key=CDS_KEY)
    print("Sistema copernicus conectado com sucesso!\n", flush=True)
except Exception as e:
    print(f"❌ Falha ao conectar com o Copernicus: {e}", flush=True)
    exit(1) # Interrompe o programa caso não consiga criar o cliente

BASE_DIR = os.getcwd()
CACHE_DIR = os.path.join(BASE_DIR, "cache_nc")
DATA_DIR = os.path.join(BASE_DIR, "data_zarr")
for d in [CACHE_DIR, DATA_DIR]: os.makedirs(d, exist_ok=True)

RANGE_ANOS = range(1990, 2026) 
AREA = [30, -100, -30, 20]

# MÓDULO 2: DICIONÁRIOS COM NOMES EXTENSOS E ASSINATURA -ERA5
# ------------------------------------------------------------------------------
VARS_ERA5_SL = {
    'total_precipitation': 'Total-Precipitation-ERA5',
    '2m_temperature': 'Temperature-At-2-Meters-ERA5',
    'sea_surface_temperature': 'Sea-Surface-Temperature-ERA5', 
    'mean_sea_level_pressure': 'Mean-Sea-Level-Pressure-ERA5',
    '10m_u_component_of_wind': 'Zonal-Wind-At-10-Meters-ERA5',
    '10m_v_component_of_wind': 'Meridional-Wind-At-10-Meters-ERA5',
    'convective_available_potential_energy': 'Convective-Available-Potential-Energy-ERA5',
    'surface_latent_heat_flux': 'Surface-Latent-Heat-Flux-ERA5',
    'surface_sensible_heat_flux': 'Surface-Sensible-Heat-Flux-ERA5',
    'total_column_water_vapour': 'Total-Column-Water-Vapour-ERA5'
}

VARS_ERA5_PL = {
    'u_component_of_wind': 'Zonal-Wind-Altitude-ERA5',
    'v_component_of_wind': 'Meridional-Wind-Altitude-ERA5',
    'specific_humidity': 'Specific-Humidity-Altitude-ERA5',
    'geopotential': 'Geopotential-Height-ERA5',
    'vertical_velocity': 'Vertical-Velocity-Omega-ERA5'
}

VARS_ORAS5 = {
    'sea_surface_temperature': 'Sea-Surface-Temperature-ORAS5',
    'mixed_layer_depth_0_01': 'Mixed-Layer-Depth-ORAS5',
    'ocean_heat_content_for_the_upper_300m': 'Ocean-Heat-Content-300-Meters-ORAS5',
    'salinity': 'Sea-Water-Salinity-ORAS5', 
    'sea_surface_height': 'Sea-Surface-Height-ORAS5'
}

# MÓDULO 3: MOTOR DE DOWNLOAD (VALIDAÇÃO DE INTEGRIDADE)
# ------------------------------------------------------------------------------
def download_engine(dataset, var_key, source, full_name, ano_ref):
    target_nc = os.path.join(CACHE_DIR, f"{source}_{full_name}_{ano_ref}.nc")
    target_zip = target_nc.replace(".nc", ".zip")
    
    if os.path.exists(target_nc):
        try:
            with xr.open_dataset(target_nc, engine='netcdf4') as tmp: return target_nc
        except Exception:
            pass
        try: os.remove(target_nc)
        except: pass

    request = {'variable': [var_key], 'year': [str(ano_ref)], 'month': [f"{m:02d}" for m in range(1, 13)], 'format': 'netcdf'}
    if 'oras5' in dataset:
        request.update({'product_type': ['consolidated'], 
                        'vertical_resolution': ['all_levels' if var_key == 'salinity' else 'single_level']})
        final_path = target_zip 
    else:
        request.update({'product_type': 'reanalysis', 'time': ['00:00', '06:00', '12:00', '18:00'], 
                        'area': AREA, 'day': [f"{d:02d}" for d in range(1, 32)]})
        if 'pressure-levels' in dataset: request.update({'pressure_level': ['850', '500', '200']})
        final_path = target_nc

    try:
        print(f"[*] Solicitando: {full_name} ({ano_ref})")
        c.retrieve(dataset, request).download(final_path)
        if final_path.endswith('.zip'):
            with zipfile.ZipFile(final_path, 'r') as z:
                ext_name = z.namelist()[0]
                z.extract(ext_name, CACHE_DIR)
                os.rename(os.path.join(CACHE_DIR, ext_name), target_nc)
            if os.path.exists(final_path): os.remove(final_path)
        return target_nc
    except Exception as e:
        print(f"❌ Erro em {full_name}: {e}"); return None

# MÓDULO 4: UNIFICAÇÃO INDEPENDENTE (ZARR CONSOLIDADO)
# ------------------------------------------------------------------------------
def create_zarr_process(files, name, ano_ref, is_oras=False):
    if not files: return None
    ds_list = []
    print(f"⚙️ Processando {name} Zarr {ano_ref}...")
    for f in files:
        ds = xr.open_dataset(f)
        if is_oras:
            t_coord = 'time_counter' if 'time_counter' in ds.coords else 'time'
            vars_to_keep = list(ds.data_vars)
            ds_d = ds[vars_to_keep].resample({t_coord: '1D'}).mean()
            if t_coord != 'time': ds_d = ds_d.rename({t_coord: 'time'})
            ds_d = ds_d.drop_vars(['nav_lat', 'nav_lon', 'deptht'], errors='ignore')
        else:
            ds = ds.drop_vars(['expver', 'number'], errors='ignore')
            if 'valid_time' in ds.coords: ds = ds.rename({'valid_time': 'time'})
            ds_d = ds.resample(time='1D').sum() if 'Precipitation' in f else ds.resample(time='1D').mean()
        
        ds_list.append(ds_d.compute())
        ds.close()

    output_path = os.path.join(DATA_DIR, f"{name}_Zarr_{ano_ref}.zarr")
    xr.merge(ds_list, compat='override').to_zarr(output_path, mode='w', consolidated=True)
    return output_path

# MÓDULO 5: AUDITORIA, RELATÓRIO TÉCNICO E LIMPEZA (SELF-CLEANING)
# ------------------------------------------------------------------------------
def get_size_gb(path):
    if not os.path.exists(path): return 0
    if os.path.isfile(path): return os.path.getsize(path) / (1024**3)
    total = 0
    for dp, dn, filenames in os.walk(path):
        for f in filenames: total += os.path.getsize(os.path.join(dp, f))
    return total / (1024**3)

def auditoria_ciclo(ano_ref, files_nc, paths_zarr):
    print(f"📊 Relatório e Auditoria - Ano {ano_ref}")
    size_zarr = sum(get_size_gb(p) for p in paths_zarr)
    
    # Critério de Auditoria: Arquivos Zarr devem existir e ter tamanho razoável
    if size_zarr > 0.1: # Exemplo: maior que 100MB
        print(f"✅ Auditoria aprovada ({size_zarr:.4f} GB). Limpando cache...")
        for f in files_nc:
            try:
                if os.path.exists(f): os.remove(f)
            except: pass
        return True
    else:
        print(f"❌ Falha na auditoria do ano {ano_ref}. Cache preservado.")
        return False

# MÓDULO 6: EXECUÇÃO DO PIPELINE (LOOP MESTRE)
# ------------------------------------------------------------------------------
global_start = time.time()

for ano in RANGE_ANOS:
    print(f"\n{'='*80}\n📅 INICIANDO PROCESSAMENTO DO ANO: {ano}\n{'='*80}")
    
    # 1. Verificação de existência (Skip if exists)
    path_e_final = os.path.join(DATA_DIR, f"ERA5_Zarr_{ano}.zarr")
    path_o_final = os.path.join(DATA_DIR, f"ORAS5_Zarr_{ano}.zarr")
    if os.path.exists(path_e_final) and os.path.exists(path_o_final):
        print(f"⏭️  Ano {ano} já concluído. Pulando para o próximo...")
        continue

    # 2. Filas de Download
    lista_era5, lista_oras5 = [], []
    for dset, dic, src in [('reanalysis-era5-single-levels', VARS_ERA5_SL, 'ERA5'),
                           ('reanalysis-era5-pressure-levels', VARS_ERA5_PL, 'ERA5')]:
        for k, v in dic.items():
            res = download_engine(dset, k, src, v, ano)
            if res: lista_era5.append(res)

    for k, v in VARS_ORAS5.items():
        res = download_engine('reanalysis-oras5', k, 'ORAS5', v, ano)
        if res: lista_oras5.append(res)

    # 3. Processamento para Zarr
    p_e = create_zarr_process(lista_era5, "ERA5", ano)
    p_o = create_zarr_process(lista_oras5, "ORAS5", ano, is_oras=True)

    # 4. Auditoria e Limpeza de Janela Deslizante
    if not auditoria_ciclo(ano, lista_era5 + lista_oras5, [p_e, p_o]):
        print("停止 - O pipeline parou para evitar perda de dados.")
        break

print(f"\n{'='*80}\n🏁 PIPELINE FINALIZADO COM SUCESSO\nTempo Total: {(time.time()-global_start)/3600:.2f}h\n{'='*80}")